# OceanCleanup System 03 — Optimization Example

This notebook adapts the multi-objective optimization workflow from `EXAMPLE_Artificial_Reef.ipynb` to the OceanCleanup System 03 offshore plastic-collection barrier. It follows the same structure:

- Define design variables and bounds
- Specify multiple objective functions and preference mappings
- Build constraints
- Run a Genetic Algorithm (GA) using two (potentially three) aggregation paradigms
- Visualize preference functions and optimization results

## Problem

System 03 is a passive, offshore plastic-collection system: a long U-shaped floating barrier towed slowly through a garbage patch by two vessels, funneling floating debris toward a retention zone at the apex. Unlike a fixed coastal structure, its design is a trade-off between capturing as much plastic as possible and avoiding harm to marine life, all while remaining operable by a small crew at reasonable cost.

Design choices — how long the barrier is, how deep its skirt hangs below the surface, how fine its screen mesh is, and how far apart the two towing vessels hold the ends — all affect capture rate, bycatch risk, structural loads, and operating cost. This notebook sets up a multi-objective optimization of those design variables, mirroring the artificial-reef example's workflow.

**Note:** the numeric bounds and reference values used below are rough starting estimates based on publicly reported figures for The Ocean Cleanup's systems — treat them as placeholders to refine with your own project research, not verified engineering specs.

## Importing Required Packages

Same dependencies as the reef example: `matplotlib`, `numpy`, `scipy` (for `pchip_interpolate` and `minimize`), and the local `genetic_algorithm_pfm` package. This notebook must stay in the same directory as `genetic_algorithm_pfm/` (i.e. inside `Civil-Engineering-Systems-Design-main/`) for the relative import to work.

In [1]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize

# Define default plotting parameters
plt.rcParams['font.size'] = '10'
plt.rcParams['savefig.dpi'] = 300

# Import local module for genetic algorithm
from genetic_algorithm_pfm import GeneticAlgorithm

## Design Variables and Bounds

The System 03 barrier geometry is parameterized using four continuous design variables:

| Variable | Description | Unit | Type |
|----------|-------------|------|------|
| `x1` | Barrier length | m | Continuous |
| `x2` | Skirt depth below waterline | m | Continuous |
| `x3` | Mesh / screen size | mm | Continuous |
| `x4` | Distance between support vessels | m | Continuous |

**Note:** as in the reef example, the code only needs each variable's bounds — not its physical meaning. `x4` (distance between vessels) is the straight-line span at the mouth of the U, so it will always be smaller than `x1` (the barrier's total length along the curve); that relationship will become a constraint in the next section, exactly like `constraint_1` did for the reef's distance-to-shore.

Bounds below are placeholder estimates — replace them with values you can justify from your own project research:

- `x1` length: longer barriers capture more plastic but cost more and are harder to tow/maintain.
- `x2` skirt depth: deeper skirts catch more submerged debris but increase drag and bycatch risk for diving animals.
- `x3` mesh size: finer mesh retains smaller plastic fragments but increases fouling, drag, and risk of trapping small marine life.
- `x4` vessel spacing: wider spacing increases the barrier's effective capture width but raises towing-force and structural demands.

`X_irl` holds a rough reference design (loosely based on publicly reported System 03 figures) for later comparison against the optimized solutions — update it once you have better sources.

In [ ]:
# Define the names of variables for later use in plotting and analysis
'''
All of those values are place holders: put in own values.
'''
design_variables = (
    ('x1', 'Barrier length',                   'm'),
    ('x2', 'Skirt depth below waterline',      'm'),
    ('x3', 'Mesh / screen size',               'mm'),
    ('x4', 'Distance between support vessels', 'm')
)

# set bounds for all variables
b1 = [500, 3000]     # x1 barrier length
b2 = [1, 6]          # x2 skirt depth
b3 = [1, 50]         # x3 mesh size
b4 = [200, 2000]     # x4 distance between vessels
bounds = [b1, b2, b3, b4]

X_irl = [2000, 4, 10, 1200] # real world values --> place holders, put in later
plot_irl = True  # Can be True or False, depending on whether you want to plot the IRL point or not